In [1]:
import os
import pandas as pd
from datetime import datetime

def ensure_validation_columns(df):
    validation_columns = {
        "Validation_Status": pd.NA,
        "Validation_Return_Pct": pd.NA,
        "Validation_Checked_On": pd.NA,
        "Validation_Age_Days": pd.NA,
    }
    for col, default in validation_columns.items():
        if col not in df.columns:
            df[col]=default
    return df


In [2]:
############################
# Fetch Stock
############################
def fetch_data(symbol):
    try:
        df = yf.download(symbol, interval=INTERVAL, period=PERIOD, progress=False)
        
        # Handle None or empty dataframe
        if df is None or df.empty:
            print(f"[WARNING] No data for {symbol}, trying alternative format...")
            # Try alternative format for NIFTY
            if symbol == "^NSEI":
                df = yf.download("NIFTY50.NS", interval=INTERVAL, period=PERIOD, progress=False)
            elif symbol.endswith(".NS"):
                # Already in NSE format
                raise ValueError(f"No data available for {symbol}")
            
            if df is None or df.empty:
                raise ValueError(f"Unable to fetch data for {symbol}")
        
        # Fix for multi-index columns from yfinance
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        
        # Ensure required columns exist
        if not all(col in df.columns for col in ['Open','High','Low','Close','Volume']):
            raise ValueError(f"Missing required columns for {symbol}")
            
        df = df[['Open','High','Low','Close','Volume']]
        df.dropna(inplace=True)
        
        if df.empty:
            raise ValueError(f"No valid data available for {symbol} after cleaning")
            
        return df
    
    except Exception as e:
        print(f"[ERROR] Failed to fetch {symbol}: {str(e)}")
        raise


In [3]:
"""
News Sentiment Engine - Refactored with Fallback
====================================
Improvements:
- FinBERT model (finance-specific NLP) with fallback
- Time-based cache with auto-expiry
- Multiple Indian market news sources (NewsAPI + RSS + FinnHub)
- Weighted scoring across sources
- Batch processing with ThreadPoolExecutor
- Graceful fallbacks
"""

import requests
import hashlib
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import feedparser
import re

# Try importing transformers, fallback to keyword-based sentiment if it fails
try:
    from transformers import pipeline
    sentiment_model = None
    print("[SENTIMENT] Loading FinBERT model...")
    try:
        sentiment_model = pipeline(
            "sentiment-analysis",
            model="ProsusAI/finbert",
            device=-1,
            truncation=True,
            max_length=512
        )
        print("[SENTIMENT] FinBERT loaded successfully.")
    except Exception as e:
        print(f"[SENTIMENT WARNING] FinBERT load failed: {e}")
        print("[SENTIMENT] Using fallback sentiment analysis...")
        sentiment_model = None
except ImportError as e:
    print(f"[WARNING] Transformers not available: {e}")
    print("[SENTIMENT] Using fallback keyword-based sentiment analysis...")
    sentiment_model = None

# ─────────────────────────────────────────────
# SIMPLE FALLBACK SENTIMENT (keyword-based)
# ─────────────────────────────────────────────

POSITIVE_KEYWORDS = {
    'bullish', 'gain', 'surge', 'rally', 'jump', 'strong', 'better', 'positive',
    'profit', 'growth', 'rise', 'boost', 'outperform', 'upgrade', 'beat', 'lead',
    'recovery', 'strength', 'momentum', 'opportunity', 'success', 'record', 'high'
}

NEGATIVE_KEYWORDS = {
    'bearish', 'fall', 'decline', 'drop', 'loss', 'weak', 'negative', 'worse',
    'crash', 'slump', 'downgrade', 'miss', 'underperform', 'risk', 'concern',
    'worry', 'weakness', 'selloff', 'tumble', 'plunge', 'pressure', 'low'
}

def _score_text_fallback(text: str) -> float:
    """Fallback sentiment scoring using keyword matching."""
    text_lower = text.lower()
    pos_count = sum(1 for word in POSITIVE_KEYWORDS if word in text_lower)
    neg_count = sum(1 for word in NEGATIVE_KEYWORDS if word in text_lower)
    
    if pos_count + neg_count == 0:
        return 0.0
    
    score = (pos_count - neg_count) / (pos_count + neg_count)
    return round(score, 3)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────

def _get_aliases(stock_name: str) -> list[str]:
    """Return search terms for a stock ticker. Handles .NS/.BSE suffix."""
    clean_ticker = stock_name.replace(".NS", "").replace(".BSE", "")
    return STOCK_ALIASES.get(clean_ticker.upper(), [clean_ticker])

# ─────────────────────────────────────────────
# HTML CLEANER
# ─────────────────────────────────────────────

def _clean_text(text: str) -> str:
    """Strip HTML tags and entities from RSS/news text."""
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'&#\d+;', "'", text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&[a-z]+;', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ─────────────────────────────────────────────
# TIME-BASED CACHE
# ─────────────────────────────────────────────

_sentiment_cache: dict = {}

def _cache_key(stock_name: str) -> str:
    return hashlib.md5(stock_name.lower().encode()).hexdigest()

def _get_cache(stock_name: str):
    key = _cache_key(stock_name)
    if key in _sentiment_cache:
        result, timestamp = _sentiment_cache[key]
        if datetime.now() - timestamp < timedelta(minutes=CACHE_EXPIRY_MINUTES):
            print(f"[CACHE HIT] {stock_name}")
            return result
    return None

def _set_cache(stock_name: str, result):
    key = _cache_key(stock_name)
    _sentiment_cache[key] = (result, datetime.now())

# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def _score_text(text: str) -> float:
    """Run FinBERT on a single text (or fallback). Returns score in [-1, +1]."""
    text = text[:SENTIMENT_TRUNCATE].strip()
    if not text:
        return 0.0
    
    # Use FinBERT if available
    if sentiment_model is not None:
        try:
            out = sentiment_model([text])[0]
            label = out["label"].lower()
            score = out["score"]
            if label == "positive":
                return round(score, 3)
            elif label == "negative":
                return round(-score, 3)
            else:
                return 0.0
        except Exception as e:
            print(f"[FINBERT ERROR] {e}")
            return _score_text_fallback(text)
    else:
        # Use keyword-based fallback
        return _score_text_fallback(text)

def _combine_title_desc(article: dict) -> str:
    title = article.get("title") or ""
    desc  = article.get("description") or ""
    return f"{title}. {desc}".strip()

# ─────────────────────────────────────────────
# SOURCE 1: NewsAPI (General + India filter)
# ─────────────────────────────────────────────

def _fetch_newsapi(stock_name: str) -> list[str]:
    global NEWSAPI_RATE_LIMITED
    if NEWSAPI_RATE_LIMITED:
        return []

    try:
        aliases   = _get_aliases(stock_name)
        query     = " OR ".join(f'"{a}"' for a in aliases)
        url = (
            f"https://newsapi.org/v2/everything"
            f"?q={requests.utils.quote(query)}"
            f"&pageSize={MAX_ARTICLES}"
            f"&language=en"
            f"&sortBy=publishedAt"
            f"&apiKey={NEWS_API_KEY}"
        )
        r = requests.get(url, timeout=5)
        if r.status_code == 429:
            print(f"[NEWSAPI] HTTP 429 - rate limit reached, disabling NewsAPI for this session")
            NEWSAPI_RATE_LIMITED = True
            return []
        if r.status_code != 200:
            print(f"[NEWSAPI] HTTP {r.status_code}")
            return []

        data     = r.json()
        articles = data.get("articles", [])

        alias_lower = [a.lower() for a in aliases]
        filtered = []
        for a in articles:
            text = _combine_title_desc(a).lower()
            if any(alias in text for alias in alias_lower):
                filtered.append(_clean_text(_combine_title_desc(a)))

        return filtered
    except Exception as e:
        print(f"[NEWSAPI ERROR] {e}")
        return []

# ─────────────────────────────────────────────
# SOURCE 2: RSS Feeds (Indian Market News)
# ─────────────────────────────────────────────

INDIAN_RSS_FEEDS = [
    "https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms",
    "https://www.moneycontrol.com/rss/marketreports.xml",
    "https://feeds.feedburner.com/ndtvprofit-latest",
]

def _fetch_rss(stock_name: str) -> list[str]:
    """Search RSS feeds for relevant headlines using aliases."""
    results     = []
    alias_lower = [a.lower() for a in _get_aliases(stock_name)]
    try:
        for feed_url in INDIAN_RSS_FEEDS:
            feed = feedparser.parse(feed_url)
            for entry in feed.entries[:30]:
                title   = _clean_text(entry.get("title", ""))
                summary = _clean_text(entry.get("summary", ""))
                combined = f"{title}. {summary}"
                if any(alias in combined[:100].lower() for alias in alias_lower):
                    results.append(combined[:SENTIMENT_TRUNCATE])
    except Exception as e:
        print(f"[RSS ERROR] {e}")
    return results[:MAX_ARTICLES]

# ─────────────────────────────────────────────
# SOURCE 3: FinnHub Company News API
# ─────────────────────────────────────────────

def _fetch_finnhub(stock_name: str) -> list[str]:
    """Fetch company news from FinnHub."""
    ticker_formats = [
        f"{stock_name.upper()}.NS",
        f"{stock_name.upper()}.BSE",
        stock_name.upper(),
    ]
    
    date_to   = datetime.now().strftime('%Y-%m-%d')
    date_from = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')
    
    for ticker in ticker_formats:
        try:
            url = (
                f"https://finnhub.io/api/v1/company-news"
                f"?symbol={ticker}"
                f"&from={date_from}"
                f"&to={date_to}"
                f"&token={FINNHUB_API_KEY}"
            )
            r = requests.get(url, timeout=5)
            if r.status_code != 200:
                continue
                
            data = r.json()
            if data and len(data) > 0:
                texts = []
                alias_lower = [a.lower() for a in _get_aliases(stock_name)]
                
                for item in data[:20]:
                    headline = _clean_text(item.get("headline", ""))
                    summary  = _clean_text(item.get("summary", ""))
                    combined = f"{headline}. {summary}"
                    
                    if any(alias in combined[:100].lower() for alias in alias_lower):
                        texts.append(combined[:SENTIMENT_TRUNCATE])
                        
                if texts:
                    print(f"[FINNHUB] {ticker} → {len(texts[:MAX_ARTICLES])} articles")
                    return texts[:MAX_ARTICLES]
                    
        except Exception as e:
            print(f"[FINNHUB ERROR] {ticker}: {e}")
            
    return []

# ─────────────────────────────────────────────
# MAIN: MULTI-SOURCE SENTIMENT FUNCTION
# ─────────────────────────────────────────────

def fetch_news_sentiment(stock_name: str) -> tuple[float, str, dict]:
    """
    Fetch and analyze news sentiment from multiple sources.

    Returns:
        (final_score, status, details)
    """

    # ── Cache check ──
    cached = _get_cache(stock_name)
    if cached:
        return cached

    sources = {
        "newsapi": lambda: _fetch_newsapi(stock_name),
        "rss": lambda: _fetch_rss(stock_name),
        "finnhub": lambda: _fetch_finnhub(stock_name),
    }

    source_texts: dict[str, list[str]] = {}
    selected_sources = {
        name: fn for name, fn in sources.items()
        if not (name == "newsapi" and NEWSAPI_RATE_LIMITED)
    }

    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {executor.submit(fn): name for name, fn in selected_sources.items()}
        for future in as_completed(futures):
            name = futures[future]
            try:
                source_texts[name] = future.result(timeout=10)
            except Exception as e:
                print(f"[SOURCE ERROR: {name}] {e}")
                source_texts[name] = []

    # Ensure fallback sources are attempted if NewsAPI is disabled
    if NEWSAPI_RATE_LIMITED:
        if "rss" not in source_texts:
            source_texts["rss"] = _fetch_rss(stock_name)
        if "finnhub" not in source_texts:
            source_texts["finnhub"] = _fetch_finnhub(stock_name)

    # ── Score each source ──
    source_weights = {
        "newsapi": 0.25,
        "rss": 0.35,
        "finnhub": 0.25
    }
    source_scores = {}
    details = {}

    for source, texts in source_texts.items():
        if not texts:
            details[source] = {"score": 0.0, "articles": 0, "status": "no_news"}
            continue

        scores = [_score_text(t) for t in texts]
        avg = round(sum(scores) / len(scores), 3)

        source_scores[source] = avg
        details[source] = {
            "score": avg,
            "articles": len(texts),
            "status": "ok",
            "headlines": [t[:100] for t in texts[:3]]
        }

    if not source_scores:
        result = (0.0, "no_news", details)
        _set_cache(stock_name, result)
        return result

    total_weight = sum(source_weights[s] for s in source_scores)
    final_score = sum(source_scores[s] * source_weights[s] for s in source_scores) / total_weight
    final_score = round(final_score, 3)
    status = "news_ok"

    result = (final_score, status, details)
    _set_cache(stock_name, result)

    print(f"[SENTIMENT] {stock_name}: score={final_score}, sources={list(source_scores.keys())}")
    return result

# ─────────────────────────────────────────────
# BATCH PROCESSING
# ─────────────────────────────────────────────

def batch_sentiment(stock_list: list[str], max_workers: int = 4) -> dict:
    """Run sentiment analysis on multiple stocks in parallel."""
    results = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(fetch_news_sentiment, stock): stock for stock in stock_list}
        for future in as_completed(futures):
            stock = futures[future]
            try:
                results[stock] = future.result()
            except Exception as e:
                print(f"[BATCH ERROR: {stock}] {e}")
                results[stock] = (0.0, "exception", {})
    return results

print("[INFO] Sentiment analysis initialized with fallback support.")

c:\Users\Gopal\anaconda3\envs\hf\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SENTIMENT] Loading FinBERT model...


Device set to use cpu


[SENTIMENT] FinBERT loaded successfully.
[INFO] Sentiment analysis initialized with fallback support.


In [4]:
import subprocess
import json

def ai_filter(stock_name, data_row, news_score):
    """
    Enhanced AI filter with:
    - News sentiment priority
    - Technical confluence scoring
    - Strong JSON validation
    - Risk override logic
    - Better error handling and logging
    - Raw output display for debugging
    """

    # Normalize news score to -1 to +1 range if needed
    try:
        news_score = float(news_score)
    except:
        news_score = 0.0

    prompt = f"""You are a professional Indian intraday trading assistant AI.

IMPORTANT RULES:
1. News sentiment has HIGH priority.
2. If NewsSentimentScore > 0.6 → Strong bullish bias unless extreme bearish technicals.
3. If NewsSentimentScore < -0.6 → Strong bearish bias unless extreme bullish technicals.
4. If NewsSentimentScore between -0.3 and 0.3 → rely mostly on technical indicators.
5. Increase confidence when: Price > VWAP, EMA9 > EMA21, RSI 55-70, High Volume
6. Decrease confidence when signals conflict.

CRITICAL: Return ONLY valid JSON, no markdown, no extra text.

Stock: {stock_name}
Price: {data_row['Close']:.2f}
VWAP: {data_row['VWAP']:.2f}
RSI: {data_row['RSI']:.1f}
Volume: {int(data_row['Volume'])}
EMA9: {data_row['EMA9']:.2f}
EMA21: {data_row['EMA21']:.2f}
ATR: {data_row['ATR']:.2f}
NewsSentimentScore: {news_score:.2f}

Return this JSON only:
{{"bias": "bullish/bearish/neutral", "confidence": <0-100>, "reason": "brief reason"}}"""

    try:
        print(f"[AI] Processing {stock_name}...", end=" ", flush=True)
        
        result = subprocess.run(
            ["ollama", "run", OLLAMA_MODEL],
            input=prompt.encode("utf-8"),
            capture_output=True,
            timeout=30,
            text=False
        )

        # ─── Check return code ───
        if result.returncode != 0:
            stderr = result.stderr.decode("utf-8", errors="ignore")
            print(f"[ERROR: ollama_error]")
            print(f"    ✗ Error Output: {stderr[:150]}")
            return {"bias": "neutral", "confidence": 0, "reason": f"ollama_error: {stderr[:50]}"}

        # ─── Decode output ───
        output = result.stdout.decode("utf-8", errors="ignore")
        output = output.replace('\r\n', '\n').replace('\r', '\n').strip()

        if not output:
            print("[ERROR: empty_output]")
            return {"bias": "neutral", "confidence": 0, "reason": "empty_output"}

        # ─── Display RAW OUTPUT ───
        print()
        print(f"    📤 RAW MODEL OUTPUT:")
        print(f"    {'-' * 60}")
        # Display in chunks to avoid truncation
        if len(output) > 300:
            print(f"    {output[:300]}...")
            print(f"    ... (total {len(output)} chars)")
        else:
            print(f"    {output}")
        print(f"    {'-' * 60}")

        # ─── Extract JSON ───
        start = output.find("{")
        end = output.rfind("}")
        
        if start == -1 or end == -1:
            print(f"    ✗ [ERROR: no_json]")
            return {"bias": "neutral", "confidence": 0, "reason": "no_json_found"}

        json_text = output[start:end+1]
        json_text = json_text.replace('\n', ' ').replace('\t', ' ')

        try:
            data = json.loads(json_text)
        except json.JSONDecodeError as e:
            # Try one more time after cleaning control characters and escaped newlines
            cleaned = json_text.replace('\\n', ' ').replace('\\r', ' ').replace('\\t', ' ')
            cleaned = ''.join(ch if ord(ch) >= 32 else ' ' for ch in cleaned)
            try:
                data = json.loads(cleaned)
            except json.JSONDecodeError as e2:
                print(f"    ✗ [ERROR: json_decode] {str(e2)[:50]}")
                print(f"    Attempted JSON: {cleaned[:200]}")
                return {"bias": "neutral", "confidence": 0, "reason": f"json_error: {str(e2)[:30]}"}

        # ─── Validate structure ───
        required_keys = {"bias", "confidence", "reason"}
        if not required_keys.issubset(data.keys()):
            missing = required_keys - data.keys()
            print(f"    ✗ [ERROR: missing_keys] {missing}")
            return {"bias": "neutral", "confidence": 0, "reason": f"missing_keys: {missing}"}

        # ─── Normalize bias ───
        if data["bias"] not in ["bullish", "bearish", "neutral"]:
            data["bias"] = "neutral"

        # ─── Validate confidence ───
        try:
            data["confidence"] = int(data["confidence"])
        except (ValueError, TypeError):
            data["confidence"] = 0

        data["confidence"] = max(0, min(100, data["confidence"]))

        # ─── News Override Logic ───
        if news_score > 0.75 and data["bias"] == "bearish":
            data["bias"] = "bullish"
            data["confidence"] = max(data["confidence"], 70)
            data["reason"] += " | strong_positive_news_override"

        if news_score < -0.75 and data["bias"] == "bullish":
            data["bias"] = "bearish"
            data["confidence"] = max(data["confidence"], 70)
            data["reason"] += " | strong_negative_news_override"

        # ─── Display PARSED JSON ───
        print(f"    ✓ [PARSED] {data['bias'][:3].upper()}/{data['confidence']}% → {data['reason'][:50]}")
        return data

    except subprocess.TimeoutExpired:
        print("[ERROR: timeout] Model took >30 seconds to respond")
        return {"bias": "neutral", "confidence": 0, "reason": "ollama_timeout"}

    except FileNotFoundError:
        print("[ERROR: ollama_not_found] Ollama executable not found in PATH")
        return {"bias": "neutral", "confidence": 0, "reason": "ollama_not_installed"}

    except Exception as e:
        print(f"[ERROR: {type(e).__name__}] {str(e)[:50]}")
        return {"bias": "neutral", "confidence": 0, "reason": f"exception: {str(e)[:30]}"}

In [5]:
############################
# OLLAMA DIAGNOSTICS (CONTINUED)
############################

import subprocess
import json

# Check 4: Test a simple prompt execution
print(f"\n[4] Testing simple prompt with '{OLLAMA_MODEL}'...")
test_prompt = "Say 'OLLAMA_TEST_OK' in JSON format: {\"status\": \"ok\"}"

try:
    print(f"    Sending prompt to model...")
    result = subprocess.run(
        ["ollama", "run", OLLAMA_MODEL],
        input=test_prompt.encode("utf-8"),
        capture_output=True,
        timeout=30,
        text=False
    )
    
    if result.returncode != 0:
        print(f"    ✗ Execution failed (return code: {result.returncode})")
        stderr = result.stderr.decode("utf-8", errors="ignore")
        print(f"    Error: {stderr[:200]}")
    else:
        output = result.stdout.decode("utf-8", errors="ignore").strip()
        
        if not output:
            print("    ✗ No output received from model")
        else:
            print(f"    ✓ Model executed successfully")
            print(f"    Raw output ({len(output)} chars):")
            print(f"    {'-' * 50}")
            print(f"    {output}")
            print(f"    {'-' * 50}")
        
except subprocess.TimeoutExpired:
    print("    ✗ Model execution timed out (30 seconds)")
except Exception as e:
    print(f"    ✗ Error testing model: {e}")

# Check 5: Test JSON parsing from output
print("\n[5] Testing JSON extraction from output...")
test_prompt_json = """Return ONLY valid JSON with no extra text:
{"test": "value", "status": "success"}"""

try:
    print(f"    Sending JSON prompt to model...")
    result = subprocess.run(
        ["ollama", "run", OLLAMA_MODEL],
        input=test_prompt_json.encode("utf-8"),
        capture_output=True,
        timeout=30,
        text=False
    )
    
    output = result.stdout.decode("utf-8", errors="ignore").strip()
    
    print(f"    Raw output ({len(output)} chars):")
    print(f"    {'-' * 50}")
    print(f"    {output}")
    print(f"    {'-' * 50}")
    
    # Try to extract JSON
    start = output.find("{")
    end = output.rfind("}")
    
    if start != -1 and end != -1:
        json_text = output[start:end+1]
        import json
        data = json.loads(json_text)
        print(f"    ✓ JSON parsing successful")
        print(f"    Extracted JSON: {data}")
    else:
        print(f"    ⚠ No JSON braces found in output")

except json.JSONDecodeError as e:
    print(f"    ⚠ JSON decoding failed: {e}")
except Exception as e:
    print(f"    ✗ Error testing JSON: {e}")

print("\n" + "=" * 60)
print("✅ OLLAMA DIAGNOSTIC COMPLETE")
print("=" * 60)

NameError: name 'OLLAMA_MODEL' is not defined

In [6]:
############################
# ALL MISSING TECHNICAL ANALYSIS FUNCTIONS
############################

# First, import required technical analysis library
import yfinance as yf
from ta.momentum import RSIIndicator, StochasticOscillator
from ta.trend import MACD, ADXIndicator, EMAIndicator
from ta.volatility import BollingerBands
from ta.volume import VolumeWeightedAveragePrice

def compute_indicators(df):
    """Calculate all required technical indicators"""
    close = df['Close']
    high = df['High']
    low = df['Low']
    volume = df['Volume']
    
    # RSI
    rsi = RSIIndicator(close, window=14)
    df['RSI'] = rsi.rsi()
    
    # Stochastic Oscillator
    stoch = StochasticOscillator(high, low, close, window=14, smooth_window=3)
    df['Stoch_K'] = stoch.stoch()
    df['Stoch_D'] = stoch.stoch_signal()
    
    # VWAP
    df['VWAP'] = VolumeWeightedAveragePrice(high, low, close, volume, window=20).volume_weighted_average_price()
    
    # EMA
    df['EMA9'] = EMAIndicator(close, window=9).ema_indicator()
    df['EMA21'] = EMAIndicator(close, window=21).ema_indicator()
    
    # ATR
    tr1 = high - low
    tr2 = abs(high - close.shift())
    tr3 = abs(low - close.shift())
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    df['ATR'] = tr.rolling(14).mean()
    
    # MACD for trend confirmation
    macd = MACD(close, window_fast=12, window_slow=26, window_sign=9)
    df['MACD'] = macd.macd()
    df['MACD_Signal'] = macd.macd_signal()
    df['MACD_Diff'] = macd.macd_diff()

    # ADX for trend strength
    adx = ADXIndicator(high, low, close, window=14)
    df['ADX'] = adx.adx()
    df['DI_Plus'] = adx.adx_pos()
    df['DI_Minus'] = adx.adx_neg()

    # Bollinger Bands
    bb = BollingerBands(close, window=20, window_dev=2)
    df['BB_Upper'] = bb.bollinger_hband()
    df['BB_Middle'] = bb.bollinger_mavg()
    df['BB_Lower'] = bb.bollinger_lband()
    
    df['AVG_VOL'] = volume.rolling(20).mean()
    return df

def compute_trend_factors(df):
    """Calculate trend factors"""
    df['EMA50'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['EMA200'] = df['Close'].ewm(span=200, adjust=False).mean()
    df['ROC10'] = df['Close'].pct_change(10)
    df['ROC20'] = df['Close'].pct_change(20)
    
    # Trend direction and strength
    df['Price_Above_EMA50'] = (df['Close'] > df['EMA50']).astype(int)
    df['Price_Above_EMA200'] = (df['Close'] > df['EMA200']).astype(int)
    
    return df

def compute_support_resistance(df):
    """Calculate recent support/resistance levels"""
    recent_high = df['High'].tail(20).max()
    recent_low = df['Low'].tail(20).min()
    
    # Pivot Point
    pivot = (df['High'].iloc[-1] + df['Low'].iloc[-1] + df['Close'].iloc[-1]) / 3
    r1 = (2 * pivot) - df['Low'].iloc[-1]
    s1 = (2 * pivot) - df['High'].iloc[-1]
    
    df['Pivot'] = pivot
    df['R1'] = r1
    df['S1'] = s1
    df['Recent_High'] = recent_high
    df['Recent_Low'] = recent_low
    
    return df

def detect_breakout(df):
    """Detect if price is breaking above recent high with volume confirmation"""
    last = df.iloc[-1]
    recent_high = df['High'].tail(20).max()
    recent_low = df['Low'].tail(20).min()
    
    breakout_up = (last['Close'] > recent_high) and (last['Volume'] > last['AVG_VOL'] * 1.5)
    breakout_down = (last['Close'] < recent_low) and (last['Volume'] > last['AVG_VOL'] * 1.5)
    
    df['Breakout_Up'] = 1 if breakout_up else 0
    df['Breakout_Down'] = 1 if breakout_down else 0
    
    return df

def detect_divergence(df):
    """Detect price-momentum divergences"""
    last_few = df.tail(5)
    
    price_higher = last_few['Close'].iloc[-1] > last_few['Close'].iloc[-3]
    rsi_lower = last_few['RSI'].iloc[-1] < last_few['RSI'].iloc[-3]
    bearish_div = price_higher and rsi_lower
    
    price_lower = last_few['Close'].iloc[-1] < last_few['Close'].iloc[-3]
    rsi_higher = last_few['RSI'].iloc[-1] > last_few['RSI'].iloc[-3]
    bullish_div = price_lower and rsi_higher
    
    df['Bearish_Div'] = 1 if bearish_div else 0
    df['Bullish_Div'] = 1 if bullish_div else 0
    
    return df

def detect_market_structure(df):
    """Detect higher highs/lows for uptrend and lower highs/lows for downtrend"""
    recent = df.tail(6)
    
    higher_highs = recent['High'].iloc[-1] > recent['High'].iloc[-2] > recent['High'].iloc[-3]
    higher_lows = recent['Low'].iloc[-1] > recent['Low'].iloc[-2] > recent['Low'].iloc[-3]
    lower_highs = recent['High'].iloc[-1] < recent['High'].iloc[-2] < recent['High'].iloc[-3]
    lower_lows = recent['Low'].iloc[-1] < recent['Low'].iloc[-2] < recent['Low'].iloc[-3]
    
    uptrend = higher_highs and higher_lows
    downtrend = lower_highs and lower_lows
    
    df['Uptrend_Structure'] = 1 if uptrend else 0
    df['Downtrend_Structure'] = 1 if downtrend else 0
    
    return df

def compute_risk_factors(df):
    """Calculate risk and liquidity factors"""
    df['Volatility20'] = df['Close'].pct_change().rolling(20).std()
    df['AvgVolume20'] = df['Volume'].rolling(20).mean()
    df['LiquidityScore'] = df['AvgVolume20'] * df['Close']
    df['Volume_Trend'] = df['Volume'] / df['AvgVolume20']
    
    return df

def compute_alpha(stock_df, index_df):
    """Calculate alpha vs index"""
    stock_ret = stock_df['Close'].pct_change(20)
    index_ret = index_df['Close'].pct_change(20)
    stock_df['Alpha'] = stock_ret - index_ret
    return stock_df

def detect_india_regime(index_df):
    """Detect market regime"""
    index_df['EMA50'] = index_df['Close'].ewm(span=50).mean()
    index_df['EMA200'] = index_df['Close'].ewm(span=200).mean()
    index_df['ROC20'] = index_df['Close'].pct_change(20)

    last = index_df.iloc[-1]

    if last['EMA50'] > last['EMA200'] and last['ROC20'] > 0:
        return "RISK_ON"
    elif last['EMA50'] < last['EMA200'] and last['ROC20'] < 0:
        return "RISK_OFF"
    else:
        return "SIDEWAYS"

def global_risk_model():
    """Assess global market risk"""
    try:
        sp500 = fetch_data("^GSPC")
        nasdaq = fetch_data("^IXIC")
        score = 0

        if sp500['Close'].pct_change(5).iloc[-1] > 0: 
            score += 1
        else: 
            score -= 1

        if nasdaq['Close'].pct_change(5).iloc[-1] > 0: 
            score += 1
        else: 
            score -= 1

        if score >= 2:
            return "GLOBAL_RISK_ON"
        elif score <= -2:
            return "GLOBAL_RISK_OFF"
        else:
            return "NEUTRAL"
    except:
        return "NEUTRAL"

def final_market_state(india_regime, global_regime):
    """Determine final market state"""
    if india_regime == "RISK_ON" and global_regime == "GLOBAL_RISK_ON":
        return "STRONG_BULL"
    elif india_regime == "RISK_OFF" and global_regime == "GLOBAL_RISK_OFF":
        return "STRONG_BEAR"
    else:
        return "MIXED"

def capital_multiplier(state):
    """Position sizing multiplier based on market state"""
    if state == "STRONG_BULL": 
        return 1.0
    elif state == "MIXED": 
        return 0.5
    else: 
        return 0.25

def score_stock(df):
    """Score stock based on multiple factors"""
    last = df.iloc[-1]
    score = 0

    # Price action relative to key trend levels
    if last['Close'] > last['VWAP']:
        score += 1.5
    if last['Close'] > last['EMA21']:
        score += 1.5
    if last['Close'] > last['EMA50']:
        score += 1.5
    if last['EMA21'] > last['EMA50']:
        score += 1.0
    if last['EMA50'] > last['EMA200']:
        score += 2.0

    # EMA confirmation
    if last['EMA9'] > last['EMA21']:
        score += 1.5
    if last['EMA21'] > last['EMA50']:
        score += 1.0
    if last['EMA9'] > last['EMA21'] and last['EMA21'] > last['EMA50']:
        score += 0.5

    # Volume confirmation
    vol_ratio = last['Volume'] / last['AVG_VOL'] if last['AVG_VOL'] > 0 else 0
    if vol_ratio > 2.0:
        score += 3.0
    elif vol_ratio > 1.5:
        score += 2.0
    elif vol_ratio > 1.2:
        score += 1.0

    if last.get('Volume_Trend', 0) > 1.2:
        score += 0.5

    # RSI analysis
    if 45 < last['RSI'] < 65:
        score += 1.5
    elif 35 < last['RSI'] <= 45:
        score += 1.0
    elif 65 <= last['RSI'] < 75:
        score += 1.0

    # ATR & Volatility
    if last['ATR'] > df['ATR'].quantile(0.6):
        score += 0.5
    if 0.02 < last.get('Volatility20', 0) < 0.06:
        score += 0.5

    # MACD confirmation
    if last['MACD'] > last['MACD_Signal']:
        score += 1.5
    if last['MACD_Diff'] > 0:
        score += 0.5
    if last['MACD'] > 0 and last['MACD_Signal'] > 0:
        score += 0.5

    # Stochastic
    if last['Stoch_K'] > last['Stoch_D'] and last['Stoch_K'] < 80:
        score += 1.0

    # ADX
    if last['ADX'] > 30:
        score += 1.5
    elif last['ADX'] > 25:
        score += 1.0
    elif last['ADX'] > 20:
        score += 0.5

    # DI+/-
    if last['DI_Plus'] > last['DI_Minus']:
        score += 1.0

    # Breakout bonus
    if last['Breakout_Up'] == 1:
        score += 2.0
    elif last['Close'] > last.get('Recent_High', last['Close']) * 0.99:
        score += 1.0

    # Structure and divergence
    if last['Uptrend_Structure'] == 1:
        score += 1.0
    if last['Bullish_Div'] == 1:
        score += 1.0
    if last['Bearish_Div'] == 1:
        score -= 2.0

    # Bollinger Bands moderation
    bb_range = last['BB_Upper'] - last['BB_Lower']
    bb_position = (last['Close'] - last['BB_Lower']) / bb_range if bb_range > 0 else 0.5
    if 0.6 < bb_position < 0.9:
        score += 0.5
    if bb_position > 0.95:
        score -= 0.5

    return score, last

def compute_factor_score(last, index_bullish):
    """Compute factor score"""
    factor_score = 0

    if last['EMA50'] > last['EMA200']:
        factor_score += 2
    if last['EMA9'] > last['EMA21']:
        factor_score += 2
    if index_bullish:
        factor_score += 1

    if last['ROC10'] > 0:
        factor_score += 1.5
    if last['ROC20'] > 0:
        factor_score += 1.5

    if last['Alpha'] > 0:
        factor_score += 1.5

    if last['Volume'] > last['AvgVolume20']:
        factor_score += 1
    if last.get('Volume_Trend', 0) > 1.1:
        factor_score += 1

    if last['Uptrend_Structure'] == 1:
        factor_score += 1
    if last['Breakout_Up'] == 1:
        factor_score += 1

    if 0.02 < last.get('Volatility20', 0) < 0.06:
        factor_score += 1

    if last['ADX'] > 25:
        factor_score += 1.5

    if last['Close'] > last['BB_Middle'] and last['Close'] < last['BB_Upper']:
        factor_score += 0.5

    return factor_score

def compute_ai_score(ai, news_score):
    """Compute AI score"""
    ai_score = 0

    if ai.get('bias') == 'bullish':
        ai_score += 3
    elif ai.get('bias') == 'bearish':
        ai_score -= 3

    if ai.get('confidence', 0) > 75:
        ai_score += 2
    elif ai.get('confidence', 0) > 50:
        ai_score += 1

    if news_score > 0.5:
        ai_score += 2
    elif news_score > 0.25:
        ai_score += 1
    elif news_score < -0.5:
        ai_score -= 2
    elif news_score < -0.25:
        ai_score -= 1

    return ((ai_score + 4) / 10) * 10

def remove_high_correlation(top_df, threshold=0.8):
    """Remove highly correlated stocks"""
    if len(top_df) <= 1:
        return top_df
    
    try:
        price_data = {}
        for stock in top_df['Stock'].head(10):
            try:
                df = fetch_data(stock + ".NS")
                price_data[stock] = df['Close'].pct_change()
            except:
                pass

        if not price_data:
            return top_df
            
        returns = pd.DataFrame(price_data)
        corr = returns.corr()

        selected = []
        for stock in corr.columns:
            if not any(abs(corr[stock][s]) > threshold for s in selected):
                selected.append(stock)

        return top_df[top_df['Stock'].isin(selected)].reset_index(drop=True)
    except:
        return top_df

def position_size(capital, volatility, multiplier):
    """Calculate position size"""
    target_risk = 0.01
    adjusted_capital = capital * multiplier
    risk_amount = adjusted_capital * target_risk
    size = risk_amount / (volatility * adjusted_capital) if (volatility * adjusted_capital) > 0 else 0
    return round(size, 4)

print("[INFO] All technical analysis functions loaded successfully!")

[INFO] All technical analysis functions loaded successfully!
